# Project: Netflix Movies and TV Shows Exploratory Data Analysis (EDA)

# Project Structure

1) Title Statement
2) Import Libraries
3) Load Data
4) Understand Data
5) Data Cleaning
6) Feature Engineering
7) Exploratory Data Analysis
8) Final Conclusion
9) GitHub ReadMe


# STEP 1: Title Statement

# Netflix Content Analysis

## Objective
Analyze Netflix Movies and TV Shows to identify content trends over time, across countries, genres, and content types to understand Netflix’s content strategy.

# STEP 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np

#visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10,6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# define color palette
palette = "viridis" 

# STEP 3: Load Data

In [ ]:
df = pd.read_csv("../data/netflix_titles.csv")

# STEP 4: Understand Data

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

# Initial data observations overview:

 - Dataset contains 8807 records and 12 features
 - Columns such as director, cast, country contain missing values
 - High cardinality columns like cast, description are not useful for analysis and can be dropped
 - Missing values in director and country can be treated as  "Unknown"
 - The date_added column should be converted to datetime format for time based analysis
 - New features such as year_added and month_added can be derived
 - The duration column contains mixed formats (minutes for movies and seasons for TV shows) and requires separation into:
 - duration_int (numeric value)
 - dutation_type (unit: min/season)


# STEP 5: Data Cleaning

In [ ]:
# convert date_added to date_time format
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

#drop columns 
df.drop(['cast','description'], axis=1, inplace = True)

#fill nulls with unknown
df.fillna({
    "director": "Unknown",
    "country": "Unknown",
    "duration": "Unknown",
    "rating" : "Unknown"
}, inplace=True)


# STEP 6: Feature Engineering

In [ ]:
# create new features

#extract year and month name
df['date_added_year'] = df['date_added'].dt.year
df['date_added_year'] = df['date_added_year'].astype('Int64')
df['date_added_month'] = df['date_added'].dt.month

# extract the time duration
df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_type'] = df['duration'].str.extract(r'([a-zA-Z]+)')
df['duration_type'] = df['duration_type'].str.lower()
df['duration_type'] = df['duration_type'].replace({'seasons': 'season'})

df['main_country'] = df['country'].str.split(',').str[0].str.strip()

# STEP 7: Exploratory Data Analysis (EDA)

# Business Questions

- What type of content dominates Netflix (Movies vs TV Shows)?
- Which countries produce the most content?
- How has Netflix content grown over the years?
- What are the most popular genres?
- What is the distribution of content duration?
  

In [ ]:
# Movies vs TV Shows countplot
sns.countplot(x='type', data = df, palette=palette)
plt.title("Distribution of Movies vs TV Shows", fontweight='bold')
plt.xlabel("Content Type")
plt.ylabel("Count")
plt.show()
### Insight: Movies dominate Netflix content compared to TV Shows, indicating a stronger focus on film-based content strategy.

# Top 10 countries
plt.figure(figsize=(8,5))
top_countries = df['main_country'].dropna().value_counts().head(10)
top_countries.sort_values().plot(kind='barh')
plt.title("Top 10 Content Producing Countries", fontweight='bold')
plt.xlabel("Number of Titles")
plt.ylabel("Country")
plt.show()
### Insight: United States leads content production, followed by India and United Kingdom.

#Content Growth Over Time
year_counts = df["date_added_year"].dropna().value_counts().sort_index()
year_counts.plot(kind='line', marker='o')
plt.title("Netflix Content Growth Over Time", fontweight='bold')
plt.xlabel("Year")
plt.ylabel("Number of Titles")
plt.grid(True)
plt.show()
### Insight: Netflix experienced rapid content growth after 2016, peaking around 2019.

#Movies duration analysis
movies = df[df['duration_type'] == 'min']
sns.histplot(movies['duration_int'], bins=30, kde=True)
plt.title("Movie Duration Distribution", fontweight='bold')
plt.xlabel("Duration (minutes)")
plt.ylabel("Frequency")
plt.show()
### Insight: Most movies are around 80–120 minutes;

#TV shows duration analysis
tv = df[df['duration_type'] == 'season']#.copy()
sns.histplot(tv['duration_int'], bins=20, kde=True)
plt.title("TV shows season Distribution", fontweight='bold')
plt.xlabel("Number of Seasons")
plt.ylabel("Frequency")
plt.show()
### Insight: Most TV shows have 1–2 seasons.

#Genre Analysis
genres = df['listed_in'].str.split(', ', expand = True).stack()
top_genres = genres.value_counts().head(10).sort_values()
top_genres.plot(kind='barh', color='coral')
plt.title("Top 10 Genres on Netflix", fontweight='bold')
plt.xlabel("Count")
plt.ylabel("Genre")
plt.show()
### Insight: International Movies and Dramas are the most common genres.

#Content added by type over time
content_trend = pd.crosstab(df['date_added_year'], df['type'])
content_trend.plot(figsize=(10,6))
plt.title("Movies vs TV Shows Added Over Time")
plt.xlabel("Year")
plt.ylabel("Count")
plt.show()

#top ratings
df['rating'].value_counts().head(10).plot(kind='bar')
plt.title("Top Content Ratings on Netflix")
plt.show()


# STEP 8: Final Conclusion

- Netflix has a stronger focus on Movies compared to TV Shows
- USA dominates content production
- Content growth peaked around 2019
- Drama and International Movies are the most popular genres
- Majority of movies fall in 90–120 minute range and TV Shows in 1-2 seasons  

# DASHBOARD

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14,10))

# Plot 1
sns.countplot(x='type', data=df, ax=axes[0,0], palette=palette)
axes[0,0].set_title("Content Type")

# Plot 2
df['main_country'].value_counts().head(5).plot(kind='barh', ax=axes[0,1])
axes[0,1].set_title("Top Countries")

# Plot 3
df['date_added_year'].value_counts().sort_index().plot(ax=axes[1,0])
axes[1,0].set_title("Growth Over Time")

# Plot 4
genres.value_counts().head(5).plot(kind='barh', ax=axes[1,1])
axes[1,1].set_title("Top Genres")

plt.tight_layout()
plt.show()